In [8]:
# ==============================================================================
# 1: 导入与全局配置
# ==============================================================================
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor  
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold
from sklearn.metrics import make_scorer
import time
def gfc_each(y_true, y_pred):
    n_t = np.linalg.norm(y_true, axis=1)
    n_p = np.linalg.norm(y_pred, axis=1)
    dot = np.sum(y_true * y_pred, axis=1)
    denominator = n_t * n_p
    return np.where(denominator == 0, 0, dot / denominator)
def gfc_mean(y_true, y_pred):
    return np.mean(gfc_each(y_true, y_pred))
scorer = make_scorer(gfc_mean, greater_is_better=True)
# ==============================================================================
# 2: 数据读取
# ==============================================================================
file_path = r"D:\project\数据整理.xlsx"
output_file = r"D:\project\ADA预测结果_GFC评分.xlsx" 
sheet_as = "AS7343去除11条的异常数据"
sheet_ev = "EVERFINE去除11条的异常数据"
df_as = pd.read_excel(file_path, sheet_name=sheet_as)
df_ev = pd.read_excel(file_path, sheet_name=sheet_ev)
ids = df_as.iloc[:, 0].values
X = df_as.iloc[:, 1:].values
Y = df_ev.iloc[:, 1:].values
# ==============================================================================
# 3: 数据集划分 
# ==============================================================================
indices = np.arange(len(ids))
stratify_labels = (ids > 343).astype(int)
X_tr, X_te, Y_tr, Y_te, idx_tr, idx_te = train_test_split(
    X, Y, indices, 
    test_size=0.2, 
    random_state=42,
    stratify=stratify_labels 
)
# ==============================================================================
# 4: 初步建模 
# ==============================================================================
base_tree_default = DecisionTreeRegressor(random_state=42, max_depth=10)
model_default = MultiOutputRegressor(
    AdaBoostRegressor(
        estimator=base_tree_default, 
        random_state=42
    )
)
model_default.fit(X_tr, Y_tr)
score_default = scorer(model_default, X_te, Y_te)
ada_default_inst = model_default.estimator
default_params_info = {
    'n_estimators': ada_default_inst.n_estimators,
    'learning_rate': ada_default_inst.learning_rate,
    'loss': ada_default_inst.loss,
}
print("\n参数组合:")
print(default_params_info)
print(f"-> 模型测试集平均 GFC: {score_default:.4f}")
# ==============================================================================
# 5: 参数寻优 
# ==============================================================================
base_tree = DecisionTreeRegressor(random_state=42, max_depth=3)
base_ada = AdaBoostRegressor(estimator=base_tree, random_state=42)
param_grid = { 
    'estimator__learning_rate': [0.1, 0.5, 1.0],       # 默认是 1.0，测试稍小的值
    'estimator__n_estimators': [50, 100, 200],         # 默认是 50，适当增加
    'estimator__loss': ['linear', 'square']            # 移除 exponential 以节省时间
}
grid_search = GridSearchCV(
    estimator=MultiOutputRegressor(base_ada),   
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, stratify_labels),
    scoring=scorer,           
    n_jobs=-1,                                            
)
grid_search.fit(X, Y)
best_params = grid_search.best_params_
best_score_cv = grid_search.best_score_
print("\n最优参数组合:")
print(best_params)
print(f"\n五折交叉验证最优平均 GFC 评分: {grid_search.best_score_:.4f}")
# ==============================================================================
# 6: 最优模型训练与评估
# ==============================================================================
ada_best_params = {}
for k, v in best_params.items():
    clean_key = k.replace('estimator__', '')
    ada_best_params[clean_key] = v
final_base_tree = DecisionTreeRegressor(random_state=42, max_depth=10)
model_final = MultiOutputRegressor(
    AdaBoostRegressor(
        estimator=final_base_tree, 
        random_state=42,
        **ada_best_params
    )
)
model_final.fit(X_tr, Y_tr)
score_best = scorer(model_final, X_te, Y_te)
print(f"-> 最优模型测试集平均 GFC: {score_best:.4f}")
# ==============================================================================
# 7: 结果输出
# ==============================================================================
pred_all = model_final.predict(X)
gfc_all = gfc_each(Y, pred_all) 
data_type_labels = ['训练集'] * len(ids)
for idx in idx_te:
    data_type_labels[idx] = '测试集'
res_df = pd.DataFrame(columns=df_ev.columns)
res_df.iloc[:, 0] = ids
res_df.iloc[:, 1:] = pred_all
res_df['GFC'] = gfc_all
res_df['数据集类型'] = data_type_labels 
os.makedirs(os.path.dirname(output_file), exist_ok=True)
res_df.to_excel(output_file, index=False)


参数组合:
{'n_estimators': 50, 'learning_rate': 1.0, 'loss': 'linear'}
-> 模型测试集平均 GFC: 0.9864

最优参数组合:
{'estimator__learning_rate': 0.1, 'estimator__loss': 'linear', 'estimator__n_estimators': 50}

五折交叉验证最优平均 GFC 评分: 0.9525
-> 最优模型测试集平均 GFC: 0.9938
